# Candidate data quality and imputation

This notebook reads only the nine Parquet files directly inside `cleaned_data`. For `Age`, `Total Assets`, and `Liabilities`, the source value `-1` is treated as missing. The original source columns remain unchanged; imputed values are written to new columns. `Criminal Cases` is excluded from imputation and all imputation comparisons.

The full-file imputed outputs use valid values from the same state/year file. Missingness diagnostics include Little's MCAR test, a permutation test, logistic regression on observed predictors, and an MNAR pattern-mixture sensitivity analysis. Criminal cases are excluded from imputation comparisons.


In [ ]:
from pathlib import Path
import math
import pandas as pd
import numpy as np

SENTINEL_NUMERIC_COLUMNS = {'Age', 'Total Assets', 'Liabilities'}
STATES = ["andhra", "bihar", "gujarat"]
YEARS = [2014, 2019, 2024]

def is_missing_value(series, column_name):
    missing = series.isna()
    if column_name in SENTINEL_NUMERIC_COLUMNS:
        numeric_values = pd.to_numeric(series, errors='coerce')
        missing = missing | numeric_values.eq(-1)
    else:
        missing = missing | series.astype(str).str.strip().eq('')
    return missing

CLEANED_DIR = Path('cleaned_data')
FULL_OUTPUT_DIR = CLEANED_DIR / 'imputed_csv_direct'
REPORT_DIR = CLEANED_DIR / 'imputation_reports'

FULL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

input_files = []
for state in STATES:
    for year in YEARS:
        input_file = CLEANED_DIR / f"{state}_{year}.parquet"
        input_files.append(input_file)

print(f'Number of state/year Parquet files: {len(input_files)}')
print(f'Full imputed files will be saved in: {FULL_OUTPUT_DIR}')
print(input_files)


## 1. Read and inspect the source files

Each input file represents one state and election year. The file name is retained in the reports and output names. The source data is loaded into memory and is never overwritten.


In [ ]:
def read_candidate_file(path):
    frame = pd.read_parquet(path)
    return frame


frames_by_file = {}

for input_path in input_files:
    frame = read_candidate_file(input_path)
    frames_by_file[input_path] = frame

    print(f'\n{input_path.name}')
    print(f'Rows: {len(frame)}')
    print(f'Columns: {frame.columns.tolist()}')

## 2. Review missingness before filling anything

This cell creates a row for every column in every file, with the missing-value count and percentage. It also saves the report as a CSV so you can inspect missingness outside the notebook.


In [ ]:
missingness_rows = []

for input_path, frame in frames_by_file.items():
    row_count = len(frame)

    for column in frame.columns:
        missing_count = int(is_missing_value(frame[column], column).sum())

        if row_count == 0:
            missing_percent = 0.0
        else:
            missing_percent = (missing_count / row_count) * 100

        missingness_rows.append(
            {
                'Source File': input_path.name,
                'Column': column,
                'Row Count': row_count,
                'Missing Count': missing_count,
                'Missing Percent': round(missing_percent, 2),
            }
        )

missingness_report = pd.DataFrame(missingness_rows)
missingness_report.to_csv(
    REPORT_DIR / 'missingness_by_state_year.csv',
    index=False,
    encoding='utf-8-sig',
)

display(missingness_report.sort_values(['Source File', 'Missing Percent'],ascending=[True, False],))

## 2a. Missingness patterns, likely mechanisms, and a diagnostic test

The count report above treats `-1` as missing for age, assets, and liabilities. This section summarizes which fields are missing together, argues plausible mechanisms, and applies the MCAR, permutation, logistic missingness, and MNAR sensitivity methods adapted from the reference notebook. No test can distinguish MAR from MNAR conclusively.


In [ ]:
MISSINGNESS_TARGET_COLUMNS = ['Age', 'Total Assets', 'Liabilities']
pattern_rows = []
missingness_masks_by_file = {}

for input_path, frame in frames_by_file.items():
    column_masks = {}

    for column in MISSINGNESS_TARGET_COLUMNS:
        if column in frame.columns:
            column_masks[column] = is_missing_value(frame[column], column)

    missingness_masks_by_file[input_path] = column_masks
    row_patterns = []

    for row_index in frame.index:
        missing_fields = []

        for column in MISSINGNESS_TARGET_COLUMNS:
            if column in column_masks and bool(column_masks[column].loc[row_index]):
                missing_fields.append(column)

        if missing_fields:
            pattern_name = ' + '.join(missing_fields)
        else:
            pattern_name = 'No target values missing'

        row_patterns.append(
            {
                'Source File': input_path.name,
                'Missing Pattern': pattern_name,
            }
        )

    file_patterns = pd.DataFrame(row_patterns)
    file_pattern_counts = (
        file_patterns.groupby(['Source File', 'Missing Pattern'])
        .size()
        .rename('Candidate Count')
        .reset_index()
    )
    pattern_rows.append(file_pattern_counts)

missingness_patterns = pd.concat(pattern_rows, ignore_index=True)
missingness_patterns.to_csv(
    REPORT_DIR / 'missingness_patterns.csv',
    index=False,
    encoding='utf-8-sig',
)

display(missingness_patterns.sort_values(
    ['Source File', 'Candidate Count'],
    ascending=[True, False],
))


### Likely missingness mechanism

Mechanisms cannot be proven from the observed data alone. The notebook records a working argument rather than treating a mechanism label as a measured fact. For age and assets, missing declarations may vary by election year, source document, or candidate record quality, which are plausible observed-data pathways for MAR. Missingness could also depend on the unobserved value itself (MNAR), such as a candidate being less likely to report a particular age or financial amount. The serial-position diagnostic checks one observable proxy only. Liabilities have no `-1` or null values in the current files, so a missingness mechanism is not applicable to that column in this run.


In [ ]:
mechanism_rows = []

for column in MISSINGNESS_TARGET_COLUMNS:
    missing_count = 0
    total_count = 0

    for input_path, frame in frames_by_file.items():
        if column in frame.columns:
            missing_count += int(
                missingness_masks_by_file[input_path][column].sum()
            )
            total_count += len(frame)

    if missing_count == 0:
        likely_mechanism = 'Not applicable: no missing values observed'
        reasoning = 'There are no null or -1 values for this field in the current Parquet files.'
    elif column == 'Age':
        likely_mechanism = 'MAR is a working assumption; MNAR remains plausible'
        reasoning = ('Age omissions may track observed source/year or record-quality differences. '
                      'If omission depends on the unobserved age itself, the process would be MNAR.')
    else:
        likely_mechanism = 'MAR is a working assumption; MNAR remains plausible'
        reasoning = ('Financial declaration omissions may track observed source/year or record-quality differences. '
                      'If reporting depends on the unobserved financial amount, the process would be MNAR.')

    mechanism_rows.append(
        {
            'Column': column,
            'Missing Count': missing_count,
            'Total Records': total_count,
            'Missing Percent': round((missing_count / total_count) * 100, 3) if total_count else 0.0,
            'Likely Mechanism / Working Assumption': likely_mechanism,
            'Argument and Limitation': reasoning,
        }
    )

mechanism_assessment = pd.DataFrame(mechanism_rows)
# mechanism_assessment.to_csv(
#     REPORT_DIR / 'missingness_mechanism_assessment.csv',
#     index=False,
#     encoding='utf-8-sig',
# )

display(mechanism_assessment)


### Test run: stratified permutation test against candidate serial position

The test asks whether missing values are associated with a candidate’s normalized serial position within its state/year file. It permutes each missingness indicator within that file, preserving the number of missing records in each file. A small p-value suggests this observed feature is associated with missingness and is inconsistent with MCAR relative to this feature; it is not proof of MAR. The test is skipped when a field has no missing values or no valid serial positions.


In [ ]:
PERMUTATION_COUNT = 4999
RANDOM_SEED = 42
random_generator = np.random.default_rng(RANDOM_SEED)

def absolute_position_difference(groups):
    missing_positions = []
    observed_positions = []

    for positions, missing_mask in groups:
        missing_positions.extend(positions[missing_mask].tolist())
        observed_positions.extend(positions[~missing_mask].tolist())

    if not missing_positions or not observed_positions:
        return np.nan

    difference = np.mean(missing_positions) - np.mean(observed_positions)
    return abs(float(difference))

test_rows = []

for column in MISSINGNESS_TARGET_COLUMNS:
    groups = []
    missing_total = 0

    for input_path, frame in frames_by_file.items():
        serial_column = 'Sno.'
        if column not in frame.columns or serial_column not in frame.columns:
            continue

        serial_values = pd.to_numeric(frame[serial_column], errors='coerce')
        missing_mask = missingness_masks_by_file[input_path][column].to_numpy(dtype=bool)
        valid_serial = serial_values.notna().to_numpy()
        if not valid_serial.any():
            continue

        normalized_position = serial_values.loc[serial_values.notna()].rank(pct=True).to_numpy()
        usable_missing_mask = missing_mask[valid_serial]
        groups.append((normalized_position, usable_missing_mask))
        missing_total += int(usable_missing_mask.sum())

    if missing_total == 0:
        test_rows.append(
            {
                'Column': column,
                'Missing Count': 0,
                'Observed Difference in Mean Position': np.nan,
                'Permutation p-value': np.nan,
                'Interpretation': 'Not applicable: no missing values to test',
            }
        )
        continue

    observed_statistic = absolute_position_difference(groups)
    exceed_count = 0

    for permutation_index in range(PERMUTATION_COUNT):
        permuted_groups = []

        for positions, missing_mask in groups:
            permuted_mask = random_generator.permutation(missing_mask)
            permuted_groups.append((positions, permuted_mask))

        permuted_statistic = absolute_position_difference(permuted_groups)
        if permuted_statistic >= observed_statistic:
            exceed_count += 1

    p_value = (exceed_count + 1) / (PERMUTATION_COUNT + 1)
    if p_value < 0.05:
        interpretation = 'Evidence of association with serial position; inconsistent with MCAR relative to this feature'
    else:
        interpretation = 'No association detected; this does not establish MCAR'

    test_rows.append(
        {
            'Column': column,
            'Missing Count': missing_total,
            'Observed Difference in Mean Position': observed_statistic,
            'Permutation p-value': p_value,
            'Interpretation': interpretation,
        }
    )

position_missingness_test = pd.DataFrame(test_rows)
# position_missingness_test.to_csv(
#     REPORT_DIR / 'missingness_position_test.csv',
#     index=False,
#     encoding='utf-8-sig',
# )

display(position_missingness_test)


### Multivariate permutation test from the reference notebook

This test shuffles each field’s missingness indicator across all candidates and checks whether the missing and observed groups differ jointly across observed features: state/year file, party, and serial position. The statistic is the sum of squared standardized mean differences. It tests association with observed information; rejection argues against MCAR but cannot distinguish MAR from MNAR.


In [ ]:
PERMUTATION_COUNT_MULTIVARIATE = 1999
multivariate_rng = np.random.default_rng(73)
observed_rows = []

for input_path, source_frame in frames_by_file.items():
    file_state_year = input_path.stem
    serial_values = pd.to_numeric(source_frame['Sno.'], errors='coerce')
    serial_rank = serial_values.rank(pct=True).fillna(0.5)

    for row_index in source_frame.index:
        row = {
            'State-Year': file_state_year,
            'Party': str(source_frame.loc[row_index, 'Party']).strip() or 'Unknown',
            'Serial Position': float(serial_rank.loc[row_index]),
        }

        for column in MISSINGNESS_TARGET_COLUMNS:
            row[f'{column}_missing'] = int(
                missingness_masks_by_file[input_path][column].loc[row_index]
            )

        observed_rows.append(row)

observed_features = pd.DataFrame(observed_rows)
party_counts = observed_features['Party'].value_counts()
common_parties = set(party_counts.head(12).index)
observed_features['Party Group'] = observed_features['Party'].where(
    observed_features['Party'].isin(common_parties),
    'Other Party',
)

categorical_features = pd.get_dummies(
    observed_features[['State-Year', 'Party Group']],
    dtype=float,
)
feature_matrix = pd.concat(
    [
        observed_features[['Serial Position']].astype(float),
        categorical_features,
    ],
    axis=1,
).to_numpy(dtype=float)
feature_variances = feature_matrix.var(axis=0)
varying_features = feature_variances > 0
feature_matrix = feature_matrix[:, varying_features]
feature_variances = feature_variances[varying_features]

def multivariate_missingness_statistic(missing_indicator, features, variances):
    missing_group = features[missing_indicator == 1]
    observed_group = features[missing_indicator == 0]

    if len(missing_group) == 0 or len(observed_group) == 0:
        return np.nan

    mean_difference = missing_group.mean(axis=0) - observed_group.mean(axis=0)
    return float(np.sum((mean_difference ** 2) / variances))

multivariate_permutation_rows = []

for column in MISSINGNESS_TARGET_COLUMNS:
    missing_indicator = observed_features[f'{column}_missing'].to_numpy(dtype=int)
    missing_count = int(missing_indicator.sum())

    if missing_count == 0:
        multivariate_permutation_rows.append(
            {
                'Column': column,
                'Missing Count': 0,
                'Observed Statistic': np.nan,
                'Permutation p-value': np.nan,
                'Interpretation': 'Not applicable: no missing values to test',
            }
        )
        continue

    observed_statistic = multivariate_missingness_statistic(
        missing_indicator,
        feature_matrix,
        feature_variances,
    )
    exceed_count = 0

    for permutation_index in range(PERMUTATION_COUNT_MULTIVARIATE):
        permuted_indicator = multivariate_rng.permutation(missing_indicator)
        permuted_statistic = multivariate_missingness_statistic(
            permuted_indicator,
            feature_matrix,
            feature_variances,
        )

        if permuted_statistic >= observed_statistic:
            exceed_count += 1

    p_value = (exceed_count + 1) / (PERMUTATION_COUNT_MULTIVARIATE + 1)
    if p_value < 0.05:
        interpretation = 'Reject MCAR relative to these observed features; MAR is plausible, MNAR remains possible'
    else:
        interpretation = 'No joint association detected; this does not establish MCAR'

    multivariate_permutation_rows.append(
        {
            'Column': column,
            'Missing Count': missing_count,
            'Observed Statistic': observed_statistic,
            'Permutation p-value': p_value,
            'Interpretation': interpretation,
        }
    )

multivariate_permutation_report = pd.DataFrame(multivariate_permutation_rows)
# multivariate_permutation_report.to_csv(
#     REPORT_DIR / 'missingness_multivariate_permutation_test.csv',
#     index=False,
#     encoding='utf-8-sig',
# )

display(multivariate_permutation_report)


## 2b. Little's MCAR test

The null hypothesis is that the numeric fields are missing completely at random. The notebook tests age, log-transformed assets, and log-transformed liabilities pooled across the nine files. The log transform reduces the extreme right skew in financial amounts. Little's test is approximate and assumes a multivariate-normal model; interpret it cautiously for these administrative data. A rejection means the data are inconsistent with MCAR, but does not distinguish MAR from MNAR.


In [ ]:
def chi_square_survival(statistic, degrees_of_freedom):
    if degrees_of_freedom <= 0 or statistic < 0:
        return np.nan

    shape = degrees_of_freedom / 2.0
    value = statistic / 2.0

    if value == 0:
        return 1.0

    epsilon = 1e-12
    tiny_value = 1e-300
    log_scale = -value + (shape * math.log(value)) - math.lgamma(shape)

    if value < shape + 1.0:
        term = 1.0 / shape
        total = term
        current_shape = shape

        for iteration in range(10000):
            current_shape += 1.0
            term *= value / current_shape
            total += term

            if abs(term) <= abs(total) * epsilon:
                break

        lower_tail = total * math.exp(log_scale)
        return min(1.0, max(0.0, 1.0 - lower_tail))

    denominator = value + 1.0 - shape
    if abs(denominator) < tiny_value:
        denominator = tiny_value

    c_value = 1.0 / tiny_value
    d_value = 1.0 / denominator
    continued_fraction = d_value

    for iteration in range(1, 10000):
        coefficient = -iteration * (iteration - shape)
        denominator += 2.0
        d_value = coefficient * d_value + denominator

        if abs(d_value) < tiny_value:
            d_value = tiny_value

        c_value = denominator + coefficient / c_value
        if abs(c_value) < tiny_value:
            c_value = tiny_value

        d_value = 1.0 / d_value
        change = d_value * c_value
        continued_fraction *= change

        if abs(change - 1.0) <= epsilon:
            break

    upper_tail = math.exp(log_scale) * continued_fraction
    return min(1.0, max(0.0, upper_tail))


def build_little_test_frame():
    numeric_parts = []

    for input_path, source_frame in frames_by_file.items():
        numeric_frame = pd.DataFrame(index=source_frame.index)

        for column in MISSINGNESS_TARGET_COLUMNS:
            numeric_values = pd.to_numeric(source_frame[column], errors='coerce')
            numeric_values = numeric_values.mask(numeric_values.eq(-1))

            if column in ['Total Assets', 'Liabilities']:
                numeric_values = np.log1p(numeric_values.where(numeric_values >= 0))

            numeric_frame[column] = numeric_values

        numeric_parts.append(numeric_frame)

    return pd.concat(numeric_parts, ignore_index=True)


def littles_mcar_test(data_frame):
    numeric_columns = data_frame.columns.tolist()
    numeric_data = data_frame[numeric_columns].copy()
    overall_means = numeric_data.mean()
    covariance = numeric_data.cov()
    missing_patterns = numeric_data.isna().astype(int)
    pattern_groups = missing_patterns.groupby(numeric_columns, dropna=False).groups

    chi_square_statistic = 0.0
    observed_dimension_total = 0
    tested_pattern_count = 0

    for pattern, row_indices in pattern_groups.items():
        observed_columns = [
            column
            for column, is_missing in zip(numeric_columns, pattern)
            if is_missing == 0
        ]

        if not observed_columns:
            continue

        pattern_data = numeric_data.loc[row_indices, observed_columns]
        pattern_means = pattern_data.mean()
        mean_difference = pattern_means - overall_means[observed_columns]
        covariance_subset = covariance.loc[observed_columns, observed_columns]
        covariance_array = covariance_subset.to_numpy(dtype=float)

        try:
            covariance_inverse = np.linalg.inv(covariance_array)
        except np.linalg.LinAlgError:
            covariance_inverse = np.linalg.pinv(covariance_array)

        difference_array = mean_difference.to_numpy(dtype=float)
        contribution = len(pattern_data) * difference_array @ covariance_inverse @ difference_array
        chi_square_statistic += float(contribution)
        observed_dimension_total += len(observed_columns)
        tested_pattern_count += 1

    degrees_of_freedom = observed_dimension_total - len(numeric_columns)
    p_value = chi_square_survival(chi_square_statistic, degrees_of_freedom)

    if degrees_of_freedom <= 0:
        status = 'Not applicable: insufficient distinct missingness patterns'
    elif p_value < 0.05:
        status = 'Reject MCAR; this does not distinguish MAR from MNAR'
    else:
        status = 'Do not reject MCAR; this does not establish MCAR'

    return {
        'Test': "Little's MCAR test",
        'Rows': len(numeric_data),
        'Incomplete Rows': int(numeric_data.isna().any(axis=1).sum()),
        'Missingness Patterns Tested': tested_pattern_count,
        'Chi-square Statistic': chi_square_statistic,
        'Degrees of Freedom': degrees_of_freedom,
        'P-value': p_value,
        'Interpretation': status,
    }


little_test_data = build_little_test_frame()
little_test_result = pd.DataFrame([littles_mcar_test(little_test_data)])
# little_test_result.to_csv(
#     REPORT_DIR / 'missingness_little_mcar_test.csv',
#     index=False,
#     encoding='utf-8-sig',
# )

display(little_test_result)

## 2c. Logistic regression for observed predictors of missingness

This adapts the reference notebook's logistic missingness model. The outcome is whether age or assets are missing. Predictors are state, election year, and normalized serial position, all observed in the candidate file name or row. The likelihood-ratio test evaluates the predictors jointly; coefficient Wald tests show which predictors are associated with missingness. A significant result is evidence against MCAR and is consistent with MAR, but it cannot rule out MNAR. A non-significant result does not prove MCAR.

The model uses a small predictor set because missing values are rare; large sets of party and education indicators can cause unstable estimates or separation in this data.


In [ ]:
def build_logistic_missingness_data():
    records = []

    for input_path, source_frame in frames_by_file.items():
        file_state, file_year_text = input_path.stem.rsplit('_', 1)
        file_year = int(file_year_text)
        serial_values = pd.to_numeric(source_frame['Sno.'], errors='coerce')
        serial_rank = serial_values.rank(pct=True)

        for column in MISSINGNESS_TARGET_COLUMNS:
            missing_mask = missingness_masks_by_file[input_path][column]

            for row_index in source_frame.index:
                if pd.isna(serial_rank.loc[row_index]):
                    continue

                records.append(
                    {
                        'Target': column,
                        'Is Missing': int(missing_mask.loc[row_index]),
                        'State': file_state,
                        'Election Year': file_year,
                        'Serial Position': float(serial_rank.loc[row_index]),
                    }
                )

    return pd.DataFrame(records)


def logistic_probability(linear_predictor):
    clipped_predictor = np.clip(linear_predictor, -35, 35)
    return 1.0 / (1.0 + np.exp(-clipped_predictor))


def logistic_log_likelihood(response, probability):
    safe_probability = np.clip(probability, 1e-12, 1.0 - 1e-12)
    return float(
        np.sum(
            response * np.log(safe_probability)
            + (1 - response) * np.log(1 - safe_probability)
        )
    )


def fit_logistic_irls(design_matrix, response):
    coefficients = np.zeros(design_matrix.shape[1], dtype=float)
    converged = False

    for iteration in range(200):
        probability = logistic_probability(design_matrix @ coefficients)
        weights = np.clip(probability * (1.0 - probability), 1e-9, None)
        gradient = design_matrix.T @ (response - probability)
        information = (design_matrix.T * weights) @ design_matrix

        try:
            step = np.linalg.solve(information, gradient)
        except np.linalg.LinAlgError:
            step = np.linalg.pinv(information) @ gradient

        coefficients = coefficients + step

        if np.max(np.abs(step)) < 1e-8:
            converged = True
            break

    probability = logistic_probability(design_matrix @ coefficients)
    information = (design_matrix.T * (probability * (1.0 - probability))) @ design_matrix
    covariance = np.linalg.pinv(information)
    return coefficients, probability, covariance, converged


def run_missingness_logistic(target_data):
    response = target_data['Is Missing'].to_numpy(dtype=float)
    missing_count = int(response.sum())

    if missing_count == 0 or missing_count == len(response):
        return {
            'Target': target_data['Target'].iloc[0],
            'Missing Count': missing_count,
            'Likelihood Ratio Statistic': np.nan,
            'Model Degrees of Freedom': np.nan,
            'Likelihood Ratio P-value': np.nan,
            'Wald Results': 'Not applicable: outcome has only one class',
            'Interpretation': 'Not applicable',
        }

    state_dummies = pd.get_dummies(target_data['State'], prefix='State', drop_first=True, dtype=float)
    year_scaled = (target_data['Election Year'].to_numpy(dtype=float) - 2014.0) / 10.0
    serial_position = target_data['Serial Position'].to_numpy(dtype=float)

    predictor_frame = state_dummies.copy()
    predictor_frame['Election Year Scaled'] = year_scaled
    predictor_frame['Serial Position'] = serial_position

    design_matrix = np.column_stack(
        [
            np.ones(len(target_data), dtype=float),
            predictor_frame.to_numpy(dtype=float),
        ]
    )

    try:
        coefficients, probability, covariance, converged = fit_logistic_irls(
            design_matrix,
            response,
        )
    except (FloatingPointError, ValueError, np.linalg.LinAlgError) as error:
        return {
            'Target': target_data['Target'].iloc[0],
            'Missing Count': missing_count,
            'Likelihood Ratio Statistic': np.nan,
            'Model Degrees of Freedom': np.nan,
            'Likelihood Ratio P-value': np.nan,
            'Wald Results': f'Model could not be estimated: {error}',
            'Interpretation': 'No conclusion; model estimation failed',
        }

    fitted_log_likelihood = logistic_log_likelihood(response, probability)
    base_rate = response.mean()
    null_probability = np.full(len(response), base_rate)
    null_log_likelihood = logistic_log_likelihood(response, null_probability)
    likelihood_ratio = max(0.0, 2.0 * (fitted_log_likelihood - null_log_likelihood))
    model_degrees_of_freedom = predictor_frame.shape[1]
    likelihood_ratio_p = chi_square_survival(likelihood_ratio, model_degrees_of_freedom)

    standard_errors = np.sqrt(np.clip(np.diag(covariance), 0.0, None))
    coefficient_labels = ['Intercept'] + predictor_frame.columns.tolist()
    wald_findings = []

    for coefficient_index, label in enumerate(coefficient_labels):
        if label == 'Intercept' or standard_errors[coefficient_index] == 0:
            continue

        wald_statistic = (coefficients[coefficient_index] / standard_errors[coefficient_index]) ** 2
        wald_p_value = chi_square_survival(wald_statistic, 1)

        if wald_p_value < 0.05:
            wald_findings.append(f'{label}: p={wald_p_value:.4g}')

    if not converged:
        interpretation = 'Model did not converge; treat coefficient results cautiously'
    elif likelihood_ratio_p < 0.05:
        interpretation = 'Observed predictors are associated with missingness; reject MCAR, MAR is plausible, MNAR remains possible'
    else:
        interpretation = 'No joint association detected; this does not establish MCAR'

    return {
        'Target': target_data['Target'].iloc[0],
        'Missing Count': missing_count,
        'Likelihood Ratio Statistic': likelihood_ratio,
        'Model Degrees of Freedom': model_degrees_of_freedom,
        'Likelihood Ratio P-value': likelihood_ratio_p,
        'Wald Results': '; '.join(wald_findings) if wald_findings else 'No individual predictor with Wald p < 0.05',
        'Interpretation': interpretation,
    }


logistic_data = build_logistic_missingness_data()
logistic_results = []

for column in MISSINGNESS_TARGET_COLUMNS:
    target_data = logistic_data[logistic_data['Target'] == column].copy()
    logistic_results.append(run_missingness_logistic(target_data))

logistic_missingness_report = pd.DataFrame(logistic_results)
# logistic_missingness_report.to_csv(
#     REPORT_DIR / 'missingness_logistic_test.csv',
#     index=False,
#     encoding='utf-8-sig',
# )

display(logistic_missingness_report)

## 2d. MNAR pattern-mixture sensitivity analysis

MNAR cannot be established from observed columns alone. This sensitivity analysis follows the reference notebook's pattern-mixture idea: for each missing field, assume the unobserved values' mean differs from the observed mean by a chosen delta, then show how the overall mean changes. Deltas are illustrative scenarios (age: ±5 years; monetary fields: ±half the observed interquartile range), not estimates learned from the missing records. Liability scenarios are skipped if the field has no missing values.


In [ ]:
def pattern_mixture_sensitivity(source_frame, input_path, column, delta_values):
    raw_values = pd.to_numeric(source_frame[column], errors='coerce')
    missing_mask = is_missing_value(source_frame[column], column)
    observed_values = raw_values.loc[~missing_mask]
    missing_count = int(missing_mask.sum())

    if missing_count == 0 or observed_values.empty:
        return []

    observed_mean = float(observed_values.mean())
    total_count = len(source_frame)
    rows = []

    for delta in delta_values:
        assumed_missing_mean = observed_mean + delta

        if column in ['Age', 'Total Assets', 'Liabilities']:
            assumed_missing_mean = max(0.0, assumed_missing_mean)

        completed_mean = (
            (observed_mean * len(observed_values))
            + (assumed_missing_mean * missing_count)
        ) / total_count

        rows.append(
            {
                'Source File': input_path.name,
                'Column': column,
                'Missing Count': missing_count,
                'Observed Mean': observed_mean,
                'Assumed Delta': delta,
                'Assumed Mean For Missing Values': assumed_missing_mean,
                'Completed-Data Mean Under Scenario': completed_mean,
            }
        )

    return rows


sensitivity_rows = []

for input_path, source_frame in frames_by_file.items():
    for column in MISSINGNESS_TARGET_COLUMNS:
        numeric_values = pd.to_numeric(source_frame[column], errors='coerce')
        numeric_values = numeric_values.mask(numeric_values.eq(-1)).dropna()

        if numeric_values.empty:
            continue

        if column == 'Age':
            delta_grid = [-5.0, 0.0, 5.0]
        else:
            interquartile_range = float(
                numeric_values.quantile(0.75) - numeric_values.quantile(0.25)
            )
            half_iqr = interquartile_range / 2.0
            delta_grid = [-half_iqr, 0.0, half_iqr]

        sensitivity_rows.extend(
            pattern_mixture_sensitivity(
                source_frame,
                input_path,
                column,
                delta_grid,
            )
        )

mnar_sensitivity_report = pd.DataFrame(sensitivity_rows)
mnar_sensitivity_report.to_csv(
    REPORT_DIR / 'mnar_pattern_mixture_sensitivity.csv',
    index=False,
    encoding='utf-8-sig',
)

display(mnar_sensitivity_report)

## 3. Imputation decisions

- `Age`, `Total Assets`, and `Liabilities`: treat both nulls and `-1` as missing, add a missingness indicator, then fill the new imputed column with the median of valid values for that state/year file. Median is less affected by very large asset and liability values than the mean.
- `Education`: retain the source values as-is; this notebook does not impute education.
- `Criminal Cases`: excluded from this imputation workflow and from its missingness and before/after comparison reports.
- `Candidate`, `Party`, constituency, serial number, and winner fields: do not impute. These describe identity, affiliation, location, or the outcome and should be checked against source records if missing.

The output CSV keeps the original columns and fills missing age, assets, and liabilities directly in those columns. The source Parquet files remain unchanged. Numeric zero is retained as a valid value.


In [ ]:
NUMERIC_IMPUTATION_COLUMNS = [
    'Age',
    'Total Assets',
    'Liabilities',
]

INDICATOR_COLUMNS = [
    'Age',
    'Total Assets',
    'Liabilities',
]

DO_NOT_IMPUTE_COLUMNS = [
    'Candidate',
    'Party',
    'Constituency',
    'Sno.',
    'Is Winner',
    'Winner Match Status',
    'Education',
]

## 4. Create full-file imputed copies for exploration

For this descriptive output, the median is calculated within each state/year file. Missing values are counted before filling. If a numeric column is entirely missing in a file, it remains missing in the imputed column and is listed in the report; the notebook does not invent a replacement value.


In [ ]:
def make_full_imputed_copy(source_frame):
    result = source_frame.copy()

    imputation_details = []

    for column in NUMERIC_IMPUTATION_COLUMNS:
        if column not in result.columns:
            continue

        numeric_values = pd.to_numeric(result[column], errors='coerce')
        numeric_values = numeric_values.mask(numeric_values.eq(-1))
        missing_count = int(is_missing_value(result[column], column).sum())

        if numeric_values.notna().any():
            median_value = numeric_values.median()
            result[column] = numeric_values.fillna(median_value)
            method = 'state_year_median'
        else:
            median_value = pd.NA
            result[column] = numeric_values
            method = 'left_missing_all_values_missing'

        imputation_details.append(
            {
                'Column': column,
                'Method': method,
                'Missing Before': missing_count,
                'Imputation Value': median_value,
            }
        )

    return result, pd.DataFrame(imputation_details)


full_imputation_summaries = []

for input_path, source_frame in frames_by_file.items():
    imputed_frame, details = make_full_imputed_copy(source_frame)
    output_path = FULL_OUTPUT_DIR / f'{input_path.stem}_imputed.csv'

    imputed_frame.to_csv(
        output_path,
        index=False,
        encoding='utf-8-sig',
    )

    details.insert(0, 'Source File', input_path.name)
    details.insert(1, 'Output File', output_path.name)
    full_imputation_summaries.append(details)

full_imputation_report = pd.concat(
    full_imputation_summaries,
    ignore_index=True,
) if full_imputation_summaries else pd.DataFrame()

full_imputation_report.to_csv(
    REPORT_DIR / 'full_file_imputation_summary.csv',
    index=False,
    encoding='utf-8-sig',
)

display(full_imputation_report)
print(f'Created {len(frames_by_file)} full-file imputed CSVs.')

## 5. Compare source and imputed values

This report compares missing counts in the source Parquet files with the remaining missing counts in the output CSVs. Criminal cases are not included in this imputation comparison.


In [ ]:
comparison_rows = []

for input_path, source_frame in frames_by_file.items():
    imputed_path = FULL_OUTPUT_DIR / f'{input_path.stem}_imputed.csv'
    imputed_frame = pd.read_csv(imputed_path)

    for column in NUMERIC_IMPUTATION_COLUMNS:
        if column not in source_frame.columns:
            continue

        original_missing_count = int(
            is_missing_value(source_frame[column], column).sum()
        )
        remaining_missing_count = int(
            is_missing_value(imputed_frame[column], column).sum()
        )
        imputed_value_count = max(
            0,
            original_missing_count - remaining_missing_count,
        )

        comparison_rows.append(
            {
                'Source File': input_path.name,
                'Column': column,
                'Original Missing Count': original_missing_count,
                'Imputed Value Count': imputed_value_count,
                'Remaining Missing Count': remaining_missing_count,
            }
        )

comparison_report = pd.DataFrame(comparison_rows)
comparison_report.to_csv(
    REPORT_DIR / 'imputation_comparison.csv',
    index=False,
    encoding='utf-8-sig',
)

display(comparison_report)
print('Imputation workflow complete. The source Parquet files were not changed.')

## 6. Compare numeric summaries before and after imputation

This comparison helps show how median filling changes the distribution. The source summary excludes missing values; the output summary uses the values after direct imputation. Use it to spot unexpected shifts before analysis.


In [ ]:
summary_rows = []

for input_path, source_frame in frames_by_file.items():
    imputed_path = FULL_OUTPUT_DIR / f"{input_path.stem}_imputed.csv"
    imputed_frame = pd.read_csv(imputed_path)

    for column in NUMERIC_IMPUTATION_COLUMNS:
        if column not in source_frame.columns or imputed_column not in imputed_frame.columns:
            continue

        original_values = pd.to_numeric(source_frame[column], errors="coerce")
        original_values = original_values.mask(original_values.eq(-1))
        completed_values = pd.to_numeric(
            imputed_frame[column],
            errors="coerce",
        )
        completed_values = completed_values.mask(completed_values.eq(-1))
        imputed_value_count = max(
            0,
            int(is_missing_value(source_frame[column], column).sum())
            - int(is_missing_value(imputed_frame[column], column).sum()),
        )
        summary_rows.append(
            {
                "Source File": input_path.name,
                "Column": column,
                "Original Nonmissing Median": original_values.median(),
                "Imputed Median": completed_values.median(),
                "Original Nonmissing Mean": original_values.mean(),
                "Imputed Mean": completed_values.mean(),
                "Original Nonmissing Count": int(original_values.notna().sum()),
                "Imputed Value Count": imputed_value_count,
                "Completed Nonmissing Count": int(completed_values.notna().sum()),
            }
        )

summary_comparison = pd.DataFrame(summary_rows)
summary_comparison.to_csv(
    REPORT_DIR / "numeric_summary_before_after.csv",
    index=False,
    encoding="utf-8-sig",
)

display(summary_comparison)
